# Replicating the Fama-French SMB and HML Factors

This notebook constructs the **SMB** (Small Minus Big) and **HML** (High Minus Low)
factors from scratch using CRSP and Compustat data via WRDS, following the exact
methodology described in:

> Fama, E. F., & French, K. R. (1993). *Common risk factors in the returns on stocks
> and bonds.* Journal of Financial Economics, 33(1), 3–56.
>
> Fama, E. F., & French, K. R. (2015). *A five-factor asset pricing model.*
> Journal of Financial Economics, 116(1), 1–22.

The constructed factors are then compared against the official factors published on
[Ken French's data library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html).

---

## Methodology overview

Every June, Fama and French sort all eligible US common stocks into **6 portfolios**
using a 2×3 independent sort on **size** (market equity) and **value** (book-to-market
ratio). The portfolios are held from **July of year $t$** through **June of year $t+1$**,
then re-sorted.

### Why June?

Most US firms have a December fiscal year-end. Accounting data for fiscal year $t-1$
is typically available by March–April of year $t$. Sorting in June ensures that
book equity is known to investors at the time of portfolio formation — avoiding
look-ahead bias. The 6-month gap (December data → June sort) is a conservative
buffer.

### The 2×3 sort

**Size breakpoint**: NYSE median of market equity (ME).
- **Small (S)**: ME ≤ NYSE median
- **Big (B)**: ME > NYSE median

**Value breakpoints**: NYSE 30th and 70th percentiles of book-to-market (BE/ME).
- **Low / Growth (L)**: BE/ME ≤ 30th percentile
- **Medium / Neutral (M)**: 30th < BE/ME ≤ 70th percentile
- **High / Value (H)**: BE/ME > 70th percentile

### Why NYSE-only breakpoints?

NASDAQ and AMEX have far more micro-cap stocks than NYSE. If breakpoints were
computed using all exchanges, the "Small" bucket would be overwhelmed by thousands
of tiny NASDAQ stocks, distorting the portfolios. Using NYSE-only breakpoints and
then *applying* them to all stocks ensures that size and value groups contain
roughly similar total market cap.

### Factor definitions

The 6 intersection portfolios are: SL, SM, SH, BL, BM, BH.

$$\text{SMB} = \frac{SL + SM + SH}{3} - \frac{BL + BM + BH}{3}$$

$$\text{HML} = \frac{SH + BH}{2} - \frac{SL + BL}{2}$$

Each portfolio return is **value-weighted** (by market equity), rebalanced annually
in June with weights updated monthly using the buy-and-hold return.

---

## Data sources

| Source | Table | What we get |
|--------|-------|-------------|
| **Compustat** | `comp.funda` | Book equity components (stockholders' equity, deferred taxes, preferred stock) |
| **CRSP** | `crsp.msf` + `crsp.msenames` | Monthly stock returns, prices, shares outstanding, exchange/share codes |
| **CRSP** | `crsp.msedelist` | Delisting returns (so we don't drop stocks that get acquired/go bankrupt) |
| **CCM** | `crsp.ccmxpf_linktable` | Links Compustat `gvkey` to CRSP `permno` (maps accounting data to market data) |
| **Ken French** | `pandas_datareader` | Official FF factor returns for comparison |

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.tseries.offsets import MonthEnd, YearEnd
from scipy import stats
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# ── Configuration ──────────────────────────────────────────────────────
START_DATE = "2000-01-01"   # Change this to go further back (Compustat starts ~1963)

# Data loading mode:
#   "wrds"  → download live from WRDS (requires username/password)
#   "local" → load from pre-saved CSVs in DATA_DIR
DATA_MODE = "wrds"
DATA_DIR = "data/"

print(f"Start date: {START_DATE}")
print(f"Data mode:  {DATA_MODE}")

## 1. Compustat: Book Equity

Book equity (BE) is constructed following Fama & French (1993, p. 8):

$$\text{BE} = \text{Stockholders' Equity} + \text{Deferred Taxes} - \text{Preferred Stock}$$

Where:
- **Stockholders' Equity** (`seq`): total shareholder equity from the balance sheet.
- **Deferred Taxes** (`txditc`): deferred taxes and investment tax credit. Added
  because it represents future tax obligations that belong to shareholders.
- **Preferred Stock** (`ps`): subtracted because preferred shareholders have priority
  over common shareholders. FF use a hierarchy: redemption value (`pstkrv`) first,
  then liquidation value (`pstkl`), then par value (`pstk`).

We require BE > 0 (negative book equity firms are dropped — they're either
distressed or have unusual accounting).

We also count the number of prior annual observations per firm. FF require
at least 2 years of Compustat data before a firm enters the sort, to avoid
backfill bias (Compustat historically added firms retroactively).

In [ ]:
if DATA_MODE == "wrds":
    import wrds
    conn = wrds.Connection()

    comp = conn.raw_sql(f"""
        SELECT gvkey, datadate, at, pstkl, txditc, pstkrv, seq, pstk
        FROM comp.funda
        WHERE indfmt = 'INDL'      -- industrial format (not financial services/utility)
          AND datafmt = 'STD'      -- standardized format
          AND popsrc = 'D'         -- domestic population
          AND consol = 'C'         -- consolidated statements
          AND datadate >= '{START_DATE}'
    """, date_cols=["datadate"])
    comp.to_csv(f"{DATA_DIR}compustat.csv", index=False)
else:
    comp = pd.read_csv(f"{DATA_DIR}compustat.csv", parse_dates=["datadate"])

comp["year"] = comp["datadate"].dt.year
print(f"Compustat: {len(comp):,} firm-years")
comp.head(3)

In [ ]:
# ── Preferred stock: use redemption value, else liquidation, else par ──
# FF (1993) p. 8: "We use redemption value... If redemption value is not
# available, we use liquidating value, and then carrying (par) value."
comp["ps"] = comp["pstkrv"].fillna(comp["pstkl"]).fillna(comp["pstk"]).fillna(0)
comp["txditc"] = comp["txditc"].fillna(0)

# ── Book equity ────────────────────────────────────────────────────────
comp["be"] = (comp["seq"] + comp["txditc"] - comp["ps"]).astype(float)
comp["be"] = np.where(comp["be"] > 0, comp["be"], np.nan)  # drop negative BE

# ── Count prior Compustat appearances (for backfill-bias filter) ──────
comp = comp.sort_values(["gvkey", "datadate"])
comp["count"] = comp.groupby("gvkey").cumcount()  # 0-indexed: count>=1 means 2+ years

comp = comp[["gvkey", "datadate", "year", "be", "count"]]
print(f"Book equity computed: {comp['be'].notna().sum():,} valid / {len(comp):,} total")

## 2. CRSP: Market Equity and Returns

From CRSP we need:
- Monthly total returns (`ret`) and ex-dividend returns (`retx`)
- Price (`prc`) and shares outstanding (`shrout`) to compute market equity
- Share code (`shrcd`) and exchange code (`exchcd`) for universe filtering

**Share codes 10 and 11** are ordinary common shares. This excludes ADRs (31),
REITs (48), closed-end funds (14), etc. — these are not part of the FF universe.

**Exchange codes 1, 2, 3** are NYSE, AMEX, and NASDAQ respectively.

### Delisting returns

When a stock is delisted (acquired, goes bankrupt, etc.), CRSP records a final
"delisting return" in `msedelist`. If we ignore this, we'd have survivorship bias —
bankrupt stocks just disappear from the sample with their last good return.
We compound the delisting return with the last regular return:

$$r_{adj} = (1 + r)(1 + r_{delist}) - 1$$

### Market equity aggregation

Some firms have multiple share classes (e.g., Google's GOOGL and GOOG). In CRSP,
each class has its own `permno` but they share a `permco` (permanent company ID).
FF use the **total ME across all share classes** for that firm. We:
1. Sum ME across all `permno`s within each `permco` per month.
2. Keep the `permno` with the largest ME as the representative security.

In [ ]:
# ── Monthly stock file ─────────────────────────────────────────────────
if DATA_MODE == "wrds":
    crsp_m = conn.raw_sql(f"""
        SELECT a.permno, a.permco, a.date, b.shrcd, b.exchcd,
               a.ret, a.retx, a.shrout, a.prc
        FROM crsp.msf AS a
        LEFT JOIN crsp.msenames AS b
            ON a.permno = b.permno
           AND b.namedt <= a.date
           AND a.date <= b.nameendt
        WHERE a.date > '{START_DATE}'
          AND b.exchcd BETWEEN 1 AND 3
    """, date_cols=["date"])
    crsp_m.to_csv(f"{DATA_DIR}crsp_monthly.csv", index=False)
else:
    crsp_m = pd.read_csv(f"{DATA_DIR}crsp_monthly.csv", parse_dates=["date"])

crsp_m[["permco", "permno", "shrcd", "exchcd"]] = (
    crsp_m[["permco", "permno", "shrcd", "exchcd"]].astype(int)
)
crsp_m["jdate"] = crsp_m["date"] + MonthEnd(0)  # align to month-end

print(f"CRSP monthly: {len(crsp_m):,} rows")

In [ ]:
# ── Delisting returns ──────────────────────────────────────────────────
if DATA_MODE == "wrds":
    dlret = conn.raw_sql("""
        SELECT permno, dlret, dlstdt
        FROM crsp.msedelist
    """, date_cols=["dlstdt"])
    dlret.to_csv(f"{DATA_DIR}crsp_delisting.csv", index=False)
else:
    dlret = pd.read_csv(f"{DATA_DIR}crsp_delisting.csv", parse_dates=["dlstdt"])

dlret["permno"] = dlret["permno"].astype(int)
dlret["jdate"] = dlret["dlstdt"] + MonthEnd(0)

# Merge delisting returns and compound with regular returns
crsp = pd.merge(crsp_m, dlret, how="left", on=["permno", "jdate"])
crsp["dlret"] = crsp["dlret"].fillna(0)
crsp["ret"] = crsp["ret"].fillna(0)
crsp["retadj"] = (1 + crsp["ret"]) * (1 + crsp["dlret"]) - 1

# ── Market equity = |price| × shares outstanding ──────────────────────
# prc is negative when CRSP uses bid-ask midpoint (no closing trade)
crsp["me"] = crsp["prc"].abs() * crsp["shrout"]
crsp = crsp.drop(["dlret", "dlstdt", "prc", "shrout"], axis=1)
crsp = crsp.sort_values(["jdate", "permco", "me"])

print(f"After delisting merge: {len(crsp):,} rows")

In [ ]:
# ── Aggregate ME across share classes within the same company ─────────
# Sum ME across all permno belonging to the same permco on a given date
crsp_summe = crsp.groupby(["jdate", "permco"])["me"].sum().reset_index()

# Keep the permno with the largest ME as the representative security
crsp_maxme = crsp.groupby(["jdate", "permco"])["me"].max().reset_index()
crsp1 = pd.merge(crsp, crsp_maxme, how="inner", on=["jdate", "permco", "me"])
crsp1 = crsp1.drop("me", axis=1)

# Replace with total company-level ME
crsp2 = pd.merge(crsp1, crsp_summe, how="inner", on=["jdate", "permco"])
crsp2 = crsp2.sort_values(["permno", "jdate"]).drop_duplicates()

print(f"After share-class aggregation: {len(crsp2):,} rows")

## 3. Portfolio Weights: The FF Weighting Scheme

FF don't just use current ME as the weight each month. They use a
**buy-and-hold adjusted weight** that starts from the June ME and evolves
with the stock's ex-dividend return over the holding year.

This is subtle but important: it means the portfolio is truly "buy and hold"
from July through June — the weight drifts with price changes, rather than
being reset to current ME each month.

The base weight for July of year $t$ is the **lagged ME from June**. For
subsequent months (August onward), the weight is:

$$w_t = \text{ME}_{\text{June}} \times \prod_{s=\text{July}}^{t-1} (1 + r^{ex}_s)$$

where $r^{ex}$ is the ex-dividend return (price return without dividends).

### December ME for BE/ME

For the book-to-market ratio, FF use **December ME** of year $t-1$ (not June ME).
This is because book equity is measured at fiscal year-end (usually December),
and using December ME in the denominator makes the ratio contemporaneous.
The June sort then uses this BE/ME ratio with a 6-month lag.

In [ ]:
crsp2["year"] = crsp2["jdate"].dt.year
crsp2["month"] = crsp2["jdate"].dt.month

# ── December ME (for BE/ME denominator) ────────────────────────────────
decme = crsp2[crsp2["month"] == 12][["permno", "me", "year"]].copy()
decme = decme.rename(columns={"me": "dec_me"})
decme["year"] = decme["year"] + 1  # will match to June of the NEXT year

# ── FF fiscal year: July=month1, ..., June=month12 ─────────────────────
crsp2["ffdate"] = crsp2["jdate"] + MonthEnd(-6)  # shift back 6 months
crsp2["ffyear"] = crsp2["ffdate"].dt.year
crsp2["ffmonth"] = crsp2["ffdate"].dt.month

# ── Buy-and-hold weights ───────────────────────────────────────────────
crsp2["1+retx"] = 1 + crsp2["retx"]
crsp2 = crsp2.sort_values(["permno", "date"])

# Cumulative ex-dividend return within each FF year (resets each July)
crsp2["cumretx"] = crsp2.groupby(["permno", "ffyear"])["1+retx"].cumprod()
crsp2["lcumretx"] = crsp2.groupby("permno")["cumretx"].shift(1)

# Lagged ME
crsp2["lme"] = crsp2.groupby("permno")["me"].shift(1)
crsp2["count"] = crsp2.groupby("permno").cumcount()
crsp2["lme"] = np.where(crsp2["count"] == 0, crsp2["me"] / crsp2["1+retx"], crsp2["lme"])

# Base ME = lagged ME as of July (ffmonth == 1)
mebase = (
    crsp2[crsp2["ffmonth"] == 1][["permno", "ffyear", "lme"]]
    .rename(columns={"lme": "mebase"})
)
crsp3 = pd.merge(crsp2, mebase, how="left", on=["permno", "ffyear"])

# Weight = base ME × cumulative ex-div return (drifts with price changes)
crsp3["wt"] = np.where(
    crsp3["ffmonth"] == 1,
    crsp3["lme"],
    crsp3["mebase"] * crsp3["lcumretx"],
)

print(f"Weights computed: {crsp3['wt'].notna().sum():,} / {len(crsp3):,}")

## 4. CRSP-Compustat Merge via CCM Link Table

CRSP identifies stocks by `permno`, Compustat identifies firms by `gvkey`.
The **CCM (CRSP-Compustat Merged)** link table maps between them.

We use links of type `L` (linking) with primary link indicators `P` or `C`.
Each link has a valid date range (`linkdt` to `linkenddt`) — we only use
the link if the June sort date falls within this range.

The Compustat fiscal year-end date is mapped to the **next June** (the sort date)
via: `jdate = yearend + 6 months`. This implements the 6-month lag that
ensures accounting data is public when used.

In [ ]:
# ── CCM link table ─────────────────────────────────────────────────────
if DATA_MODE == "wrds":
    ccm = conn.raw_sql("""
        SELECT gvkey, lpermno AS permno, linktype, linkprim, linkdt, linkenddt
        FROM crsp.ccmxpf_linktable
        WHERE SUBSTR(linktype, 1, 1) = 'L'
          AND (linkprim = 'C' OR linkprim = 'P')
    """, date_cols=["linkdt", "linkenddt"])
    ccm.to_csv(f"{DATA_DIR}ccm_link.csv", index=False)
else:
    ccm = pd.read_csv(f"{DATA_DIR}ccm_link.csv", parse_dates=["linkdt", "linkenddt"])

ccm["linkenddt"] = ccm["linkenddt"].fillna(pd.to_datetime("today"))

# ── Merge Compustat book equity with CCM links ─────────────────────────
ccm1 = pd.merge(comp[["gvkey", "datadate", "be", "count"]], ccm, how="left", on="gvkey")
ccm1["yearend"] = ccm1["datadate"] + YearEnd(0)
ccm1["jdate"] = ccm1["yearend"] + MonthEnd(6)  # fiscal year-end → next June

# Keep only links valid at the June sort date
ccm2 = ccm1[(ccm1["jdate"] >= ccm1["linkdt"]) & (ccm1["jdate"] <= ccm1["linkenddt"])]
ccm2 = ccm2[["gvkey", "permno", "datadate", "yearend", "jdate", "be", "count"]]

print(f"CCM-linked firm-years: {len(ccm2):,}")

In [ ]:
# ── June snapshot: merge CRSP market data with Compustat book equity ───
crsp3_jun = crsp3[crsp3["month"] == 6]
crsp_jun = pd.merge(crsp3_jun, decme, how="inner", on=["permno", "year"])
crsp_jun = crsp_jun[
    ["permno", "date", "jdate", "shrcd", "exchcd", "retadj",
     "me", "wt", "cumretx", "mebase", "lme", "dec_me"]
].sort_values(["permno", "jdate"]).drop_duplicates()

# Link with book equity
ccm_jun = pd.merge(crsp_jun, ccm2, how="inner", on=["permno", "jdate"])

# ── Book-to-market ratio ───────────────────────────────────────────────
# BE is in millions (Compustat), ME is in thousands (CRSP: prc × shrout)
ccm_jun["beme"] = ccm_jun["be"] * 1000 / ccm_jun["dec_me"]

print(f"June panel (CRSP+Compustat): {len(ccm_jun):,} firm-years")

## 5. NYSE Breakpoints and Portfolio Assignment

Universe filters (applied to **NYSE stocks** for breakpoints, and to
**all stocks** for portfolio membership):

- `shrcd ∈ {10, 11}` — ordinary common shares only
- `exchcd ∈ {1, 2, 3}` — NYSE, AMEX, NASDAQ (already filtered in CRSP query)
- `beme > 0` and `me > 0` — positive market equity and book-to-market
- `count >= 1` — at least 2 years of prior Compustat data (backfill-bias filter)

In [ ]:
# ── NYSE breakpoints ───────────────────────────────────────────────────
# Only NYSE (exchcd==1) common stocks (shrcd 10 or 11) with valid data
nyse = ccm_jun[
    (ccm_jun["exchcd"] == 1)
    & (ccm_jun["beme"] > 0)
    & (ccm_jun["me"] > 0)
    & (ccm_jun["count"] >= 1)
    & (ccm_jun["shrcd"].isin([10, 11]))
]

# Size: NYSE median ME
nyse_sz = (
    nyse.groupby("jdate")["me"].median()
    .rename("size_median")
    .reset_index()
)

# Value: NYSE 30th and 70th percentile of BE/ME
nyse_bm = (
    nyse.groupby("jdate")["beme"]
    .describe(percentiles=[0.3, 0.7])
    .reset_index()[["jdate", "30%", "70%"]]
    .rename(columns={"30%": "bm30", "70%": "bm70"})
)

nyse_breaks = pd.merge(nyse_sz, nyse_bm, on="jdate")

# Apply breakpoints to ALL stocks (not just NYSE)
ccm1_jun = pd.merge(ccm_jun, nyse_breaks, how="left", on="jdate")

print(f"Breakpoints for {len(nyse_breaks)} June dates")
nyse_breaks.tail(3)

In [ ]:
# ── Portfolio assignment ───────────────────────────────────────────────
# Eligibility mask: same filters as NYSE but applied to all exchanges
mask = (
    (ccm1_jun["beme"] > 0)
    & (ccm1_jun["me"] > 0)
    & (ccm1_jun["count"] >= 1)
)
eligible = ccm1_jun[mask].copy()

# Size bucket
eligible["szport"] = np.where(eligible["me"] <= eligible["size_median"], "S", "B")

# Value bucket
eligible["bmport"] = np.where(
    eligible["beme"] <= eligible["bm30"], "L",
    np.where(eligible["beme"] <= eligible["bm70"], "M", "H"),
)

# Store June assignments
june = eligible[["permno", "jdate", "szport", "bmport"]].copy()
june["ffyear"] = june["jdate"].dt.year

# Count by portfolio
june["port"] = june["szport"] + june["bmport"]
print("Stocks per portfolio (pooled across years):")
print(june["port"].value_counts().sort_index())

## 6. Compute Value-Weighted Monthly Returns

In [ ]:
# ── Map June assignments to all monthly returns (July t → June t+1) ───
crsp3_cols = [
    "date", "permno", "shrcd", "exchcd", "retadj",
    "me", "wt", "cumretx", "ffyear", "jdate",
]
ccm3 = pd.merge(
    crsp3[crsp3_cols],
    june[["permno", "ffyear", "szport", "bmport"]],
    how="left",
    on=["permno", "ffyear"],
)

# Keep only eligible observations
ccm4 = ccm3[
    (ccm3["wt"] > 0)
    & ccm3["szport"].notna()
    & ccm3["bmport"].notna()
    & ccm3["shrcd"].isin([10, 11])
].copy()

print(f"Monthly obs with portfolio assignments: {len(ccm4):,}")

In [ ]:
# ── Value-weighted returns per portfolio ───────────────────────────────
def wavg(group, ret_col, wt_col):
    """Weighted average return."""
    d, w = group[ret_col], group[wt_col]
    try:
        return (d * w).sum() / w.sum()
    except ZeroDivisionError:
        return np.nan


vwret = (
    ccm4.groupby(["jdate", "szport", "bmport"])
    .apply(wavg, "retadj", "wt", include_groups=False)
    .reset_index()
    .rename(columns={0: "vwret"})
)
vwret["port"] = vwret["szport"] + vwret["bmport"]

# Pivot to wide format: columns = BH, BL, BM, SH, SL, SM
ff = vwret.pivot(index="jdate", columns="port", values="vwret")

print(f"Monthly portfolio returns: {len(ff)} months")
ff.head(3)

In [ ]:
# ── Construct SMB and HML ─────────────────────────────────────────────
ff["SMB"] = (ff["SL"] + ff["SM"] + ff["SH"]) / 3 - (ff["BL"] + ff["BM"] + ff["BH"]) / 3
ff["HML"] = (ff["SH"] + ff["BH"]) / 2 - (ff["SL"] + ff["BL"]) / 2

our_factors = ff[["SMB", "HML"]].dropna()
our_factors.index.name = "date"

print(f"Factors constructed: {len(our_factors)} months")
print(f"Period: {our_factors.index.min().date()} to {our_factors.index.max().date()}")
our_factors.describe().round(4)

## 7. Comparison with Official Fama-French Factors

We download the official factors from
[Ken French's data library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html)
and compare using:

1. **Pearson correlation** — measures linear co-movement.
2. **OLS regression** (Our = α + β × Official) — β near 1 means matching scale;
   α near 0 means no systematic bias.
3. **Engle-Granger cointegration test** — checks whether the cumulative factor
   series share a long-run equilibrium (they don't drift apart permanently).

In [ ]:
import pandas_datareader.data as web

ff_official = web.DataReader(
    "F-F_Research_Data_5_Factors_2x3", "famafrench",
    start=START_DATE, end="2026",
)[0] / 100  # percent → decimal
ff_official.index = ff_official.index.to_timestamp()

# Align dates
our = our_factors.copy()
our.index = pd.to_datetime(our.index).to_period("M").to_timestamp()

ff_off = ff_official[["SMB", "HML"]].copy()
ff_off.index = pd.to_datetime(ff_off.index).to_period("M").to_timestamp()

comp_df = (
    our.rename(columns={"SMB": "Our SMB", "HML": "Our HML"})
    .join(ff_off.rename(columns={"SMB": "FF SMB", "HML": "FF HML"}), how="inner")
    .dropna()
)

print(f"Overlapping months: {len(comp_df)}")
print(f"Period: {comp_df.index.min().date()} to {comp_df.index.max().date()}")

In [ ]:
from statsmodels.tsa.stattools import coint
import statsmodels.api as sm


def compare_factor(df, our_col, ff_col, name):
    """Full statistical comparison: correlation, regression, cointegration."""
    y, x = df[our_col].values, df[ff_col].values

    corr, corr_p = stats.pearsonr(y, x)

    X_ols = sm.add_constant(x)
    model = sm.OLS(y, X_ols).fit()
    alpha, beta = model.params[0], model.params[1]
    r2 = model.rsquared

    coint_stat, coint_p, crit = coint(y, x)

    print(f"\n{'=' * 60}")
    print(f"  {name}: Our Construction vs Official FF")
    print(f"{'=' * 60}")
    print(f"  Pearson Correlation:  {corr:.4f}  (p = {corr_p:.2e})")
    print(f"  Regression Alpha:     {alpha:.6f}")
    print(f"  Regression Beta:      {beta:.4f}")
    print(f"  R-squared:            {r2:.4f}")
    print(f"  Cointegration:        stat = {coint_stat:.4f}, p = {coint_p:.4f}")
    print(f"    Critical: 1%={crit[0]:.2f}, 5%={crit[1]:.2f}, 10%={crit[2]:.2f}")


compare_factor(comp_df, "Our SMB", "FF SMB", "SMB")
compare_factor(comp_df, "Our HML", "FF HML", "HML")

## 8. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for i, (name, our_col, ff_col) in enumerate([
    ("SMB", "Our SMB", "FF SMB"),
    ("HML", "Our HML", "FF HML"),
]):
    # Cumulative returns
    ax = axes[i, 0]
    cum_our = (1 + comp_df[our_col]).cumprod()
    cum_ff = (1 + comp_df[ff_col]).cumprod()
    ax.plot(comp_df.index, cum_our, label=f"Our {name}", linewidth=2)
    ax.plot(comp_df.index, cum_ff, label=f"Official {name}",
            linestyle="--", linewidth=2, alpha=0.8)
    ax.set_title(f"Cumulative Returns: {name}")
    ax.set_ylabel("Growth of $1")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Scatter plot
    ax = axes[i, 1]
    ax.scatter(comp_df[ff_col], comp_df[our_col], alpha=0.3, s=10)
    lo = min(comp_df[ff_col].min(), comp_df[our_col].min())
    hi = max(comp_df[ff_col].max(), comp_df[our_col].max())
    ax.plot([lo, hi], [lo, hi], "r--", alpha=0.5, label="45° line")
    ax.set_xlabel(f"Official {name}")
    ax.set_ylabel(f"Our {name}")
    ax.set_title(f"{name}: Scatter (Our vs Official)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── 60-month rolling correlation ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i, (name, our_col, ff_col) in enumerate([
    ("SMB", "Our SMB", "FF SMB"),
    ("HML", "Our HML", "FF HML"),
]):
    rcorr = comp_df[our_col].rolling(60).corr(comp_df[ff_col])
    ax = axes[i]
    ax.plot(comp_df.index, rcorr, linewidth=1.5)
    ax.axhline(0.9, color="green", ls="--", alpha=0.5, label="0.9")
    ax.axhline(0.8, color="orange", ls="--", alpha=0.5, label="0.8")
    ax.set_title(f"{name}: 60-Month Rolling Correlation")
    ax.set_ylabel("Correlation")
    ax.set_ylim(0.4, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Performance Summary

In [ ]:
summary = pd.DataFrame(columns=["Mean", "Vol", "Sharpe", "Max Drawdown"])

for col in ["Our SMB", "FF SMB", "Our HML", "FF HML"]:
    r = comp_df[col].dropna()
    cum = (1 + r).cumprod()
    dd = (cum / cum.cummax() - 1).min()
    summary.loc[col] = [
        r.mean(),
        r.std(),
        r.mean() / r.std() if r.std() != 0 else np.nan,
        dd,
    ]

display(summary.map(lambda x: f"{x:.4f}"))

In [ ]:
# ── Tracking error ────────────────────────────────────────────────────
for name in ["SMB", "HML"]:
    diff = comp_df[f"Our {name}"] - comp_df[f"FF {name}"]
    print(f"{name}:")
    print(f"  Annualized tracking error:  {diff.std() * np.sqrt(12):.4f}")
    print(f"  Annualized mean difference: {diff.mean() * 12:.4f}")
    print()

## 10. Sources of Difference

Our replication will not match the official FF factors exactly. Known sources
of difference include:

1. **Book equity definition**: FF use a specific hierarchy with multiple
   fallbacks for stockholders' equity (SEQ → CEQ+PSTK → AT-LT). We use
   the SEQ path only, which covers the vast majority of observations.

2. **Weight updating**: FF update portfolio weights within the holding year
   using cumulative ex-dividend returns on the *total firm ME* (aggregated
   across share classes). Small differences in how share-class aggregation
   interacts with the weight update can accumulate.

3. **Data vintage**: Compustat backfills and corrects historical data.
   The version of Compustat available today may differ slightly from what
   was available when FF originally computed their factors.

4. **Delisting treatment**: the exact handling of partial-month delisting
   returns and the timing of when a delisted stock exits the portfolio
   can differ.

A correlation above **0.95 for SMB** and **0.90 for HML** indicates an
excellent replication.

In [ ]:
# ── Close WRDS connection if open ─────────────────────────────────────
if DATA_MODE == "wrds":
    conn.close()
    print("WRDS connection closed.")